In [15]:
import json
import os
import ast

In [16]:
# Get list of all files in the directory
files = os.listdir('./Filtered_Output/')
jsonl_files = [file for file in files if file.endswith('.jsonl') and file.startswith('multi-')]
len(jsonl_files)

24

In [17]:
def check_compilable(data):
    try:
        ast.parse(data)
        return True
    except:
        return False

In [18]:
compilation_results = {}
for file in jsonl_files:
    print(file)
    with open(f'./Filtered_Output/{file}', 'r') as f:
        data = [json.loads(line) for line in f]

    model_name = file.split('_')[1]
    temp = file.split('_')[2].replace('.jsonl', '')
    print(f"Model Name: {model_name}, Temp: {temp}")
    if model_name not in compilation_results:
        compilation_results[model_name] = {}
    if temp not in compilation_results[model_name]:
        compilation_results[model_name][temp] = {}

    for i in range(len(data)):
        language = data[i]['language']

        if language not in compilation_results[model_name][temp]:
            compilation_results[model_name][temp][language] = {"Total": 0,
            "Compilable_before": 0,
            "Compilable_after": 0,
        }
        for j in range(len(data[i]['output'])):
            compilation_results[model_name][temp][language]["Total"] += 1

            old_code = data[i]['output'][j]['code']
            lines = old_code.split('\n')
            if "```python" in lines[0]:
                    lines = lines[1:]
            if "```" in lines[-1]:
                    lines = lines[:-1]
            old_code = "\n".join(lines)
            if check_compilable(old_code):
                compilation_results[model_name][temp][language]["Compilable_before"] += 1

            cleared_code = data[i]['output'][j]['cleared_code']
            if data[i]['output'][j]["compilable"] == True:
                compilation_results[model_name][temp][language]["Compilable_after"] += 1

multi-dataset_gpt-4o-mini_0.0.jsonl
Model Name: gpt-4o-mini, Temp: 0.0
multi-dataset_gpt-4o-mini_0.2.jsonl
Model Name: gpt-4o-mini, Temp: 0.2
multi-dataset_gpt-4o-mini_0.6.jsonl
Model Name: gpt-4o-mini, Temp: 0.6
multi-dataset_gpt-4o-mini_0.4.jsonl
Model Name: gpt-4o-mini, Temp: 0.4
multi-dataset_gpt-4o-mini_1.0.jsonl
Model Name: gpt-4o-mini, Temp: 1.0
multi-dataset_gemini-2.5-flash_1.0.jsonl
Model Name: gemini-2.5-flash, Temp: 1.0
multi-dataset_starcoder2_0.0.jsonl
Model Name: starcoder2, Temp: 0.0
multi-dataset_gemini-2.5-flash_0.4.jsonl
Model Name: gemini-2.5-flash, Temp: 0.4
multi-dataset_Qwen_0.2.jsonl
Model Name: Qwen, Temp: 0.2
multi-dataset_Qwen_0.0.jsonl
Model Name: Qwen, Temp: 0.0
multi-dataset_gemini-2.5-flash_0.6.jsonl
Model Name: gemini-2.5-flash, Temp: 0.6
multi-dataset_starcoder2_0.2.jsonl
Model Name: starcoder2, Temp: 0.2
multi-dataset_Qwen_1.0.jsonl
Model Name: Qwen, Temp: 1.0
multi-dataset_starcoder2_0.6.jsonl
Model Name: starcoder2, Temp: 0.6
multi-dataset_Qwen_0.4.j

In [19]:
compilation_data = []
for model_name, temp_data in compilation_results.items():
    for temp, language_data in temp_data.items():
        for language, counts in language_data.items():
            compilation_data.append({
                "Model": model_name,
                "Temp": temp,
                "Language": language,
                "Total": counts["Total"],
                "Compilable_before (%)": (counts["Compilable_before"]/ counts["Total"]) * 100,
                "Compilable_after (%)": (counts["Compilable_after"]/ counts["Total"]) * 100,
            })

In [22]:
import pandas as pd 
df = pd.DataFrame(compilation_data, columns=["Model", "Temp", "Language", "Total", "Compilable_before (%)", "Compilable_after (%)"])
# Sort the DataFrame by Model, Temp, and Language
df = df.sort_values(by=["Model", "Temp", "Language"])

df.to_csv('compilation_results_multi.csv', index=False)